# Bounding Box Review

Review and filter detections per panel **before** running `extract_crops.py`.

**Workflow:**
1. Pick a panel from the dropdown  
2. Adjust the containment threshold — sub-crops (bboxes mostly inside a larger one) are auto-excluded (red)  
3. Toggle individual detections in the card grid to manually include/exclude  
4. Click **Save approved** — writes `annotated/<panel>_approved.json` (same schema, excluded detections removed)  
5. Run `extract_crops.py --approved-dir` to crop only approved detections

Approved files accumulate — you can review panels in any order.

In [ ]:
# ── Cell 1: imports & paths ────────────────────────────────────────────────
import json
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

REPO_ROOT    = Path("../..")
ANNOTATED    = REPO_ROOT / "frobenius_artifacts/analysis/annotated"
PANELS_DIR   = REPO_ROOT / "frobenius_artifacts/analysis/panels"

# Collect all panels that have both a detections JSON and a panel PNG
_jsons = sorted(ANNOTATED.glob("*_detections.json"))
_panels = []
for j in _jsons:
    stem = j.stem.replace("_detections", "")
    png  = PANELS_DIR / f"{stem}.png"
    if png.exists():
        _panels.append((stem, j, png))
    else:
        # try _cropped suffix variant
        alt = PANELS_DIR / f"{stem}_cropped.png"
        if alt.exists():
            _panels.append((stem, j, alt))

print(f"{len(_panels)} panels with both detections JSON and panel PNG")
for stem, j, png in _panels[:5]:
    dets = json.loads(j.read_text())
    print(f"  {stem}: {len(dets)} detections")

In [ ]:
# ── Cell 2: main review UI ─────────────────────────────────────────────────
#
# Controls:
#   Panel dropdown         — pick a panel to review
#   Containment threshold  — fraction of smaller bbox inside larger to auto-exclude
#   Min IoU                — auto-exclude low-confidence detections
#   Detection cards        — thumbnail + metadata + Include checkbox per detection
#   Save approved button   — writes annotated/<panel>_approved.json

# ── Containment helpers ───────────────────────────────────────────────────
def _containment(a, b):
    """Fraction of smaller bbox area covered by intersection of a and b."""
    ax1, ay1 = a["x"], a["y"]
    ax2, ay2 = ax1 + a["w"], ay1 + a["h"]
    bx1, by1 = b["x"], b["y"]
    bx2, by2 = bx1 + b["w"], by1 + b["h"]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    min_area = min(a["w"] * a["h"], b["w"] * b["h"])
    return inter / min_area if min_area > 0 else 0.0


def _auto_exclude(dets, threshold):
    """Return set of indices auto-excluded by containment filtering."""
    n = len(dets)
    areas = [d["bbox"]["w"] * d["bbox"]["h"] for d in dets]
    by_size = sorted(range(n), key=lambda i: -areas[i])
    suppress = set()
    for pos, i in enumerate(by_size):
        if i in suppress:
            continue
        for j in by_size[pos + 1:]:
            if j in suppress:
                continue
            if _containment(dets[i]["bbox"], dets[j]["bbox"]) >= threshold:
                suppress.add(j)
    return suppress


# ── Panel image drawing ───────────────────────────────────────────────────
MAX_DISPLAY_W = 900   # px — scale large panels down for display

def _draw_panel(panel_img, dets, include_mask, out):
    """Render panel with colour-coded bounding boxes into out (Output widget)."""
    iw, ih = panel_img.size
    scale = min(1.0, MAX_DISPLAY_W / iw)
    dw, dh = int(iw * scale), int(ih * scale)

    fig, ax = plt.subplots(figsize=(dw / 96, dh / 96), dpi=96)
    ax.imshow(panel_img.resize((dw, dh), Image.LANCZOS))
    ax.axis("off")

    for i, d in enumerate(dets):
        b = d["bbox"]
        x, y, w, h = b["x"] * scale, b["y"] * scale, b["w"] * scale, b["h"] * scale
        included = include_mask[i]
        color = "#44dd44" if included else "#ff4444"
        lw    = 2.5 if included else 1.5
        alpha = 0.9 if included else 0.55
        rect = mpatches.Rectangle(
            (x, y), w, h,
            linewidth=lw, edgecolor=color, facecolor="none", alpha=alpha
        )
        ax.add_patch(rect)
        # index label
        ax.text(
            x + 3, y + 3,
            str(d["index"]),
            fontsize=max(6, int(10 * scale)),
            color=color, va="top", fontweight="bold",
            bbox=dict(facecolor="black", alpha=0.4, pad=1, linewidth=0),
        )

    plt.tight_layout(pad=0)
    out.clear_output(wait=True)
    with out:
        plt.show()
    plt.close(fig)


# ── State ─────────────────────────────────────────────────────────────────
_state = {
    "stem": None,
    "dets": [],
    "panel_img": None,
    "checkboxes": [],   # one per detection
    "auto_excl": set(),
}


# ── Widgets ───────────────────────────────────────────────────────────────
_sl = dict(continuous_update=False,
           style={"description_width": "180px"},
           layout=widgets.Layout(width="60%"))

w_panel = widgets.Dropdown(
    options=[(s, i) for i, (s, _, __) in enumerate(_panels)],
    description="Panel:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="90%"),
)
w_contain = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.05, value=0.75,
    description="containment threshold",
    readout_format=".2f", **_sl,
)
w_min_iou = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.05, value=0.0,
    description="min predicted_iou",
    readout_format=".2f", **_sl,
)
btn_save   = widgets.Button(
    description="Save approved",
    button_style="success",
    layout=widgets.Layout(width="160px", margin="8px 0"),
)
btn_all_in  = widgets.Button(description="Include all",  button_style="info",
                              layout=widgets.Layout(width="120px"))
btn_all_out = widgets.Button(description="Exclude all",  button_style="warning",
                              layout=widgets.Layout(width="120px"))
btn_reset   = widgets.Button(description="Reset to auto", button_style="",
                              layout=widgets.Layout(width="130px"))

out_panel  = widgets.Output()   # panel image
out_cards  = widgets.Output()   # detection card grid
out_status = widgets.Output()   # save status


def _include_mask():
    return [cb.value for cb in _state["checkboxes"]]


def _redraw_panel():
    if _state["panel_img"] is None:
        return
    _draw_panel(_state["panel_img"], _state["dets"], _include_mask(), out_panel)


def _build_cards():
    """Build a grid of detection cards (thumbnail + metadata + checkbox)."""
    dets      = _state["dets"]
    panel_img = _state["panel_img"]
    auto_excl = _state["auto_excl"]
    checkboxes = []

    THUMB = 96
    cards = []
    for i, d in enumerate(dets):
        b = d["bbox"]
        # crop from panel image
        iw, ih = panel_img.size
        x1 = max(0, b["x"])
        y1 = max(0, b["y"])
        x2 = min(iw, b["x"] + b["w"])
        y2 = min(ih, b["y"] + b["h"])
        crop = panel_img.crop((x1, y1, x2, y2))
        crop.thumbnail((THUMB, THUMB))
        sq = Image.new("RGB", (THUMB, THUMB), (30, 30, 30))
        sq.paste(crop, ((THUMB - crop.width)//2, (THUMB - crop.height)//2))

        # Convert PIL → PNG bytes for widget
        import io
        buf = io.BytesIO()
        sq.save(buf, format="PNG")
        thumb_w = widgets.Image(value=buf.getvalue(), format="png",
                                width=THUMB, height=THUMB)

        auto_tag = " ⚠ sub-crop" if i in auto_excl else ""
        cb = widgets.Checkbox(
            value=(i not in auto_excl),
            description=f"Include",
            indent=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="100px"),
        )
        checkboxes.append(cb)
        cb.observe(lambda _: _redraw_panel(), names="value")

        iou  = d.get("predicted_iou", d.get("pred_iou", "?"))
        stab = d.get("stability_score", "?")
        area = d.get("area_ratio", "?")
        meta = widgets.HTML(
            f"<div style='font-size:11px;line-height:1.5;color:#ccc'>"
            f"<b>#{d['index']}</b> {d.get('scale','?')}<br>"
            f"area: {area:.3f}<br>"
            f"iou: {iou:.3f}<br>"
            f"stab: {stab:.3f}<br>"
            f"{b['w']}×{b['h']} px"
            f"<span style='color:#ff8888'>{auto_tag}</span>"
            f"</div>"
        )
        card = widgets.VBox(
            [thumb_w, meta, cb],
            layout=widgets.Layout(
                border="1px solid #333",
                padding="4px",
                margin="3px",
                width="120px",
                background="#1a1a1a",
            )
        )
        cards.append(card)

    _state["checkboxes"] = checkboxes

    COLS = 6
    rows = []
    for r in range(0, len(cards), COLS):
        rows.append(widgets.HBox(cards[r:r+COLS]))

    out_cards.clear_output(wait=True)
    with out_cards:
        display(widgets.VBox(rows))


def _load_panel(_=None):
    idx = w_panel.value
    stem, json_path, png_path = _panels[idx]
    dets      = json.loads(json_path.read_text())
    panel_img = Image.open(png_path).convert("RGB")

    # apply min_iou pre-filter
    min_iou = w_min_iou.value
    dets = [d for d in dets
            if d.get("predicted_iou", d.get("pred_iou", 1.0)) >= min_iou]

    auto_excl = _auto_exclude(dets, w_contain.value)

    _state.update(stem=stem, dets=dets, panel_img=panel_img, auto_excl=auto_excl)

    _build_cards()
    _redraw_panel()

    out_status.clear_output()
    with out_status:
        n_auto = len(auto_excl)
        print(f"{stem} — {len(dets)} detections, "
              f"{n_auto} auto-excluded as sub-crops")


def _on_threshold(_=None):
    """Recompute containment filter and rebuild cards (preserves manual overrides)."""
    dets = _state["dets"]
    if not dets:
        return
    # remember current manual state
    prev = _include_mask()
    auto_excl = _auto_exclude(dets, w_contain.value)
    _state["auto_excl"] = auto_excl
    _build_cards()
    # restore manual overrides where they existed
    for i, cb in enumerate(_state["checkboxes"]):
        if i < len(prev):
            cb.value = prev[i]
        else:
            cb.value = (i not in auto_excl)
    _redraw_panel()


def _include_all(_):
    for cb in _state["checkboxes"]:
        cb.value = True


def _exclude_all(_):
    for cb in _state["checkboxes"]:
        cb.value = False


def _reset_to_auto(_):
    auto = _state["auto_excl"]
    for i, cb in enumerate(_state["checkboxes"]):
        cb.value = (i not in auto)


def _save(_):
    dets = _state["dets"]
    stem = _state["stem"]
    mask = _include_mask()
    approved = [d for d, inc in zip(dets, mask) if inc]
    out_path = ANNOTATED / f"{stem}_approved.json"
    out_path.write_text(json.dumps(approved, indent=2))
    out_status.clear_output()
    with out_status:
        print(f"Saved {len(approved)}/{len(dets)} approved detections → {out_path.name}")


w_panel.observe(_load_panel, names="value")
w_contain.observe(_on_threshold, names="value")
w_min_iou.observe(_load_panel, names="value")
btn_save.on_click(_save)
btn_all_in.on_click(_include_all)
btn_all_out.on_click(_exclude_all)
btn_reset.on_click(_reset_to_auto)

display(
    widgets.HTML("<h3 style='margin:4px 0'>Bounding Box Review</h3>"),
    w_panel,
    widgets.HBox([w_contain, w_min_iou]),
    widgets.HBox([btn_all_in, btn_all_out, btn_reset, btn_save]),
    out_status,
    out_panel,
    widgets.HTML("<b style='font-size:13px'>Detection cards — toggle to include/exclude</b>"),
    out_cards,
)
_load_panel()